# Live runs on Gemma

Gemma is Google's open release, and it is here for a reason the other five
cannot supply. Paired with `gemini-3.5-flash-lite` it puts the same lab on both
sides of the weights split: one model where Google controls how it is served and
one where it does not. Whether a vendor's age behaviour changes when it loses
that control is a question this panel can now ask and no existing child safety
benchmark has.

It is reached through a local Ollama daemon relaying to ollama.com, which is what
the `-cloud` suffix in the identifier names. The daemon holds the signed in
session, so no key is sent from here. Requests go out through the same
`build_payload` every other model uses, so the panel's reasoning and sampling
settings apply to this arm as they do to the rest.

There is no batch queue, so the pass runs live in five parts. Each part is
checkpointed, read straight into the results, and reports before the next begins.
Interrupting is safe: every reply is written as it arrives, and re-running a part
asks only for what that part still lacks.

Billed by subscription rather than by token, so the cost meter stays silent for
this model. The thing to watch instead is the quota, which this arm shares with
the classifier.

In [1]:
# Import the libraries
import json
import sys
from pathlib import Path
import pandas as pd

In [2]:
# Set the working directory to the project root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

/Users/rinlobachevskii/Desktop/Git/Thesis


In [3]:
# Import the pipeline
%load_ext autoreload
%autoreload 2

import backends
import run
import settings
import utils

needs = {'run': ['generate_part', 'join_parts', 'read_batch', 'part_path',
                 'set_aside_replies'],
         'utils': ['api_key', 'read_lines', 'read_table', 'result_path'],
         'backends': ['USAGE', 'spent', 'record_usage', 'call_api'],
         'settings': ['MODELS', 'GENERATION', 'BATCHES_DIR']}
missing = [f'{name}.{attr}' for name, attrs in needs.items()
           for attr in attrs if not hasattr(globals()[name], attr)]
if missing:
    raise SystemExit('Scripts are out of date, missing: ' + ', '.join(missing)
                     + '\nCopy scripts/ from the latest package and restart the kernel.')

utils.make_directories()
pd.set_option('display.max_colwidth', 70)
print('Scripts are current')

Scripts are current


## The model

In [ ]:
MODEL = 'gemma4:31b-cloud'
PARTS = 5

spec = next(e for e in settings.MODELS.values() if e['id'] == MODEL)
prompts = utils.read_table(settings.PROMPTS_PATH)
wanted = len(prompts) * settings.GENERATION['replicates']
have = len(utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR)))

print(f'Model      {MODEL} on {spec["provider"]}')
print(f'Reached at  {backends.OLLAMA_URL}')
print(f'Billed      by subscription, so no per token cost is recorded')
print(f'Cap         {settings.GENERATION["max_tokens"]} tokens, '
      f'temperature {settings.GENERATION["temperature"]}')
print(f'Collected   {have:,} of {wanted:,}, in {PARTS} parts of '
      f'{-(-wanted // PARTS):,}')

## Rerunning

`FRESH` moves an earlier pass to `results/superseded/` and asks for every prompt
again. Leave it false to finish a pass that stopped part way, which is the
normal case here since a live run of four thousand calls will not always
complete in one sitting.

In [ ]:
FRESH = False        # True only when a request parameter has changed

if FRESH:
    moved = run.set_aside_replies(MODEL)
    print(f'Earlier pass set aside at {moved}' if moved
          else 'Nothing collected yet, so nothing to set aside')
else:
    print('Normal run: only what is missing will be requested')

## What it will take

No per token price, so nothing to estimate in money. The constraint is the
subscription quota, which this arm shares with the classifier that has 66,000
classifications to do. Run one part, then look at the console before committing
the other four.

In [ ]:
# There is no per token price for this model, so nothing to project. What is
# worth knowing before a full pass is how long it takes and how much quota it
# uses, which one part will tell you.
left = wanted - have
print(f'{left:,} calls outstanding, about {-(-left // PARTS):,} a part')
print('Run one part, then check the quota in the Ollama console before the rest.')

## Generate, one part at a time

Each part is its own cell, so a part that finishes is banked whatever happens to
the next one. Run them in order, or re-run any single one: a part already
collected reports nothing outstanding rather than being asked for again.

Requests go out several at a time. A live call spends nearly all of its time
waiting rather than sending, so this finishes in a fraction of the time and
costs exactly the same.

In [ ]:
# Define once, then run each part below. Re-running a part asks only for what
# that part still lacks, so an interrupted part costs nothing but its own time.
totals = {'read': 0, 'failed': 0, 'truncated': 0, 'repeated': 0,
          'blocked': 0, 'input': 0, 'output': 0, 'cost': 0.0}


def run_part(part):
    path, asked, failures = run.generate_part(MODEL, part, PARTS)
    if not asked:
        print(f'Part {part} of {PARTS}: nothing outstanding')
        return
    # a part where every call failed writes no file, so there is nothing to read
    if not path.exists():
        print(f'Part {part} of {PARTS}: all {asked:,} calls failed, nothing '
              f'written. Fix the cause and run this cell again.')
        return

    # counted from the ingest rather than the generation, so that a response
    # recorded on the way out and again on the way in is not billed twice
    backends.USAGE.update(calls=0, input=0, output=0)
    read, failed, truncated, repeated, blocked = run.read_batch(MODEL, path)
    usage = dict(backends.USAGE)
    for name, value in [('read', read), ('failed', failed),
                        ('truncated', truncated), ('repeated', repeated),
                        ('blocked', blocked), ('input', usage['input']),
                        ('output', usage['output'])]:
        totals[name] += value

    print(f'\nPart {part} of {PARTS}')
    print(f'Read {read:,} replies, {failed} failed, {truncated} truncated, '
          f'{repeated:,} already had')
    print(f'Tokens: {usage["input"]:,} input, {usage["output"]:,} output')
    print(f'Output tokens a reply: {usage["output"] / max(read - failed, 1):.0f}')


print(f'{PARTS} parts of {-(-wanted // PARTS):,}, '
      f'{utils.WORKERS} requests in flight at a time')

In [8]:
run_part(1)

  deepseek-v4-flash part 1  96 of 864, 5594 an hour, 0.1 hours left, 0 failed
     $0.0102 spent, $0.09 projected for this pass, 41,252 tokens
  deepseek-v4-flash part 1  132 of 864, 3688 an hour, 0.2 hours left, 0 failed
     $0.0168 spent, $0.11 projected for this pass, 66,466 tokens
  deepseek-v4-flash part 1  204 of 864, 3784 an hour, 0.2 hours left, 0 failed
     $0.0284 spent, $0.12 projected for this pass, 111,665 tokens
  deepseek-v4-flash part 1  264 of 864, 3664 an hour, 0.2 hours left, 0 failed
     $0.0382 spent, $0.13 projected for this pass, 149,816 tokens
  deepseek-v4-flash part 1  348 of 864, 3684 an hour, 0.1 hours left, 0 failed
     $0.0515 spent, $0.13 projected for this pass, 201,723 tokens
  deepseek-v4-flash part 1  372 of 864, 3325 an hour, 0.1 hours left, 0 failed
     $0.0576 spent, $0.13 projected for this pass, 224,713 tokens
  deepseek-v4-flash part 1  456 of 864, 3515 an hour, 0.1 hours left, 0 failed
     $0.0710 spent, $0.13 projected for this pass, 276

In [9]:
run_part(2)

  deepseek-v4-flash part 2  132 of 864, 7646 an hour, 0.1 hours left, 0 failed
     $0.1465 spent, $0.96 projected for this pass, 573,393 tokens
  deepseek-v4-flash part 2  228 of 864, 6274 an hour, 0.1 hours left, 0 failed
     $0.1590 spent, $0.60 projected for this pass, 623,104 tokens
  deepseek-v4-flash part 2  264 of 864, 4915 an hour, 0.1 hours left, 0 failed
     $0.1660 spent, $0.54 projected for this pass, 649,825 tokens
  deepseek-v4-flash part 2  336 of 864, 4643 an hour, 0.1 hours left, 0 failed
     $0.1753 spent, $0.45 projected for this pass, 686,467 tokens
  deepseek-v4-flash part 2  420 of 864, 4711 an hour, 0.1 hours left, 0 failed
     $0.1883 spent, $0.39 projected for this pass, 737,263 tokens
  deepseek-v4-flash part 2  480 of 864, 4490 an hour, 0.1 hours left, 0 failed
     $0.1981 spent, $0.36 projected for this pass, 775,529 tokens
  deepseek-v4-flash part 2  588 of 864, 4737 an hour, 0.1 hours left, 0 failed
     $0.2092 spent, $0.31 projected for this pass, 

In [ ]:
run_part(3)

  deepseek-v4-flash part 3  168 of 864, 9579 an hour, 0.1 hours left, 0 failed
     $0.1359 spent, $0.70 projected for this pass, 537,046 tokens
  deepseek-v4-flash part 3  276 of 864, 7935 an hour, 0.1 hours left, 0 failed
     $0.1469 spent, $0.46 projected for this pass, 581,824 tokens
  deepseek-v4-flash part 3  372 of 864, 7171 an hour, 0.1 hours left, 0 failed
     $0.1601 spent, $0.37 projected for this pass, 633,844 tokens
  deepseek-v4-flash part 3  456 of 864, 6557 an hour, 0.1 hours left, 0 failed
     $0.1734 spent, $0.33 projected for this pass, 685,453 tokens
  deepseek-v4-flash part 3  552 of 864, 6301 an hour, 0.0 hours left, 0 failed
     $0.1849 spent, $0.29 projected for this pass, 731,521 tokens
  deepseek-v4-flash part 3  660 of 864, 6294 an hour, 0.0 hours left, 0 failed
     $0.1973 spent, $0.26 projected for this pass, 781,305 tokens
  deepseek-v4-flash part 3  732 of 864, 6021 an hour, 0.0 hours left, 0 failed
     $0.2072 spent, $0.24 projected for this pass, 

In [ ]:
run_part(4)

In [ ]:
run_part(5)

## Join and total

Run once every part is done. The parts are joined into one file and removed,
leaving a single record of what the provider returned.

In [ ]:
joined, lines = run.join_parts(MODEL, PARTS)
print(f'Joined {lines:,} responses into {joined.name}, part files removed')

print(f'\nAll parts')
print(f'Read {totals["read"]:,} replies, {totals["failed"]} failed, '
      f'{totals["truncated"]} truncated, {totals["repeated"]:,} already had, '
      f'{totals["blocked"]} blocked')
print(f'Tokens: {totals["input"]:,} input, {totals["output"]:,} output')
print(f'Output tokens a reply: '
      f'{totals["output"] / max(totals["read"] - totals["failed"], 1):.0f}')

## Check what arrived

In [ ]:
# Bring the two flags up to date from the raw provider file, then report.
# Safe to re-run: it recomputes from data/batches/ rather than accumulating.
import flags
flags.apply(MODEL)
collected = utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR))
prompts = utils.read_table(settings.PROMPTS_PATH)

if collected.empty:
    print(f'Nothing collected for {MODEL} yet')
else:
    marked = lambda name: collected.get(name, pd.Series('', index=collected.index)) \
        .astype(str).str.strip().isin(['True', 'true']) \
        | ~collected.get(name, pd.Series('', index=collected.index)) \
        .astype(str).str.strip().isin(['', 'False', 'false', 'nan'])
    blank = int((collected['response'].astype(str).str.strip() == '').sum())
    errored = int((collected['error'].astype(str).str.strip() != '').sum())
    print(f'Replies: {len(collected):,}, {blank} empty, {errored} errored, '
          f'{int(marked("blocked").sum())} blocked, '
          f'{int(marked("truncated").sum())} truncated')
    print(f"Coverage: {collected['prompt_id'].nunique():,} of {len(prompts):,} "
          f"prompts")

    display(collected.merge(prompts[['prompt_id', 'condition', 'prompt']],
                            on='prompt_id')[['condition', 'prompt',
                                             'response']].head(5))